# 03 · From HTML to clean text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pretraining-data/03_text_extraction.ipynb) &nbsp; 📖 Concept deck: [Text extraction](https://nilesh-salpe.github.io/visual-ai/pretraining-data/05-text-extraction/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** stripping scripts, styles and navigation from HTML, splitting it into text blocks, and dropping boilerplate blocks with link-density and length heuristics, the core idea behind extractors like jusText and trafilatura.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import re, html as _html
PAGE = """<html><head><title>Ring all-reduce explained</title><style>.nav{color:red}</style>
<script>var tracking = "<p>not text</p>";</script></head><body>
<div class="nav"><a href="/">Home</a> | <a href="/blog">Blog</a> | <a href="/about">About</a> | <a href="/login">Log in</a></div>
<h1>Ring all-reduce explained</h1>
<p>Data-parallel training averages gradients across GPUs. A ring all-reduce does this in two phases, a reduce-scatter and an all-gather.</p>
<p>Each GPU sends about twice the gradient size in total, almost independently of the number of GPUs, which is why the method scales.</p>
<div class="share"><a href="/tw">Tweet</a> <a href="/fb">Share</a></div>
<p>We use cookies to improve your experience. <a href="/privacy">Privacy policy</a></p>
<footer><a href="/terms">Terms</a> &copy; 2024 Example Inc.</footer>
</body></html>"""

## Task 1: Remove scripts, styles and comments

Remove `<script>…</script>`, `<style>…</style>` and `<!-- … -->` (including their contents, across lines). Return the remaining HTML. Note the script contains a fake `<p>` that must not survive.

In [ ]:
def strip_noise(html_text):
    # TODO: regex with re.S | re.I
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_strip_noise(f):
    out = f(PAGE)
    assert "tracking" not in out and "not text" not in out and ".nav{" not in out
    assert "Ring all-reduce explained" in out

check("strip_noise", _test_strip_noise, strip_noise)

## Task 2: Split into text blocks with link density

Split the cleaned HTML at block-level tags (`p`, `div`, `h1`–`h6`, `li`, `footer`, `header`, `title`). For each block return `(text, link_density)` where text has tags removed, entities unescaped and whitespace collapsed, and link density = characters inside `<a>…</a>` ÷ all characters. Skip empty blocks.

In [ ]:
def blocks(clean_html):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_blocks(f):
    b = f(re.sub(r"<script.*?</script>|<style.*?</style>", " ", PAGE, flags=re.S | re.I))
    texts = [t for t, _ in b]
    nav = [d for t, d in b if t.startswith("Home")]
    assert nav and nav[0] > 0.6, "the navigation bar is almost all link text"
    body = [d for t, d in b if t.startswith("Data-parallel")]
    assert body and body[0] == 0
    assert any("©" in t for t in texts), "entities like &copy; must be unescaped"

check("blocks", _test_blocks, blocks)

## Task 3: Keep the main content

Keep blocks with link density below `max_ld`, at least `min_words` words, and no boilerplate phrase (`cookie`, `privacy policy`, `terms of use`, `©`). Keep the `<h1>`-style headline too: allow blocks of ≥ 3 words if they are the first block after the title. Return the kept texts joined with blank lines.

In [ ]:
def extract(html_text, max_ld=0.3, min_words=8):
    # TODO: strip_noise → blocks → filter
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_extract(f):
    out = f(PAGE)
    assert "reduce-scatter" in out and "twice the gradient" in out, "keep the article paragraphs"
    for junk in ["Log in", "Tweet", "cookies", "Terms"]:
        assert junk not in out, f"boilerplate '{junk}' should be removed"

check("extract", _test_extract, extract)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Remove scripts, styles and comments
def strip_noise_solution(html_text):
    return re.sub(r"<script.*?</script>|<style.*?</style>|<!--.*?-->", " ", html_text, flags=re.S | re.I)

check("strip_noise (reference)", _test_strip_noise, strip_noise_solution)

In [ ]:
# @title Solution for Task 2: Split into text blocks with link density
def blocks_solution(clean_html):
    parts = re.split(r"</?(?:p|div|h[1-6]|li|footer|header|title)\b[^>]*>", clean_html, flags=re.I)
    out = []
    for part in parts:
        link = "".join(re.findall(r"<a\b[^>]*>(.*?)</a>", part, flags=re.S | re.I))
        text = re.sub(r"\s+", " ", _html.unescape(re.sub(r"<[^>]+>", " ", part))).strip()
        if not text: continue
        link_text = re.sub(r"\s+", " ", _html.unescape(re.sub(r"<[^>]+>", " ", link))).strip()
        out.append((text, len(link_text) / len(text)))
    return out

check("blocks (reference)", _test_blocks, blocks_solution)

In [ ]:
# @title Solution for Task 3: Keep the main content
def extract_solution(html_text, max_ld=0.3, min_words=8):
    bad = ("cookie", "privacy policy", "terms of use", "©")
    b = blocks_solution(strip_noise_solution(html_text)); keep = []
    for i, (t, ld) in enumerate(b):
        if ld >= max_ld or any(x in t.lower() for x in bad): continue
        if len(t.split()) >= min_words or (i <= 2 and len(t.split()) >= 3 and i > 0):
            keep.append(t)
    return "\n\n".join(dict.fromkeys(keep))

check("extract (reference)", _test_extract, extract_solution)

---
## Experiments
**How much of the page is kept?** Compare the number of characters in the raw HTML, the naive tag-stripped text (what WET files roughly give you) and the extracted main content.

In [ ]:
naive = re.sub(r"\s+", " ", _html.unescape(re.sub(r"<[^>]+>", " ", PAGE)))
main = extract_solution(PAGE)
print(f"raw HTML {len(PAGE)} chars → naive text {len(naive)} chars → main content {len(main)} chars")
print("naive text starts:", naive[:120])

---
## Takeaways
* Raw HTML is mostly markup, scripts and boilerplate; naive tag stripping keeps menus, buttons and cookie banners.
* Block-level heuristics (link density, length, boilerplate phrases) recover the main text cheaply.
* FineWeb found extracting from WARC with trafilatura beat Common Crawl's WET text, which keeps too much boilerplate.